# Forest Segmentation — SegFormer-B0 From Scratch (Colab)

Same pipeline as the pretrained baseline, but MiT-B0 encoder + decode head are
**randomly initialized** (no ImageNet weights).

This notebook reports the full **Table 2** row for scratch SegFormer-B0:
Dice, IoU, F1, Precision, Params, GFLOPs.

**Before running (Colab):** Runtime → Change runtime type → GPU (T4).


## Step 0: Environment

On Colab: mounts Drive. Locally: skips mount and uses `Phase1/Kalana-Person2`.


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab: Drive mounted")
else:
    print("Local run: skipping Google Drive mount")


Mounted at /content/drive
Colab: Drive mounted


In [2]:
import sys, subprocess
pkgs = ["transformers", "accelerate", "tqdm", "thop"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")


deps ready


## Step 1: Config + verify filename pairing


In [3]:
import os
import sys
from pathlib import Path

# ── Config ────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Kalana")
    CHECKPOINT_OUT = Path("/content/drive/MyDrive/segformer_b0_scratch.pt")
    RESULTS_DIR = Path("/content/drive/MyDrive/segformer_scratch_results")
else:
    DATA_ROOT = Path(".").resolve()
    if not (DATA_ROOT / "images").is_dir():
        DATA_ROOT = Path("Phase1/Kalana-Person2").resolve()
    CHECKPOINT_OUT = DATA_ROOT / "checkpoints" / "segformer_b0_scratch.pt"
    RESULTS_DIR = DATA_ROOT / "results"
    (DATA_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)

IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
LR = 6e-5
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
# If a checkpoint already exists, skip training and only evaluate Table 2 metrics
EVAL_ONLY = CHECKPOINT_OUT.exists()
# ──────────────────────────────────────────────────────────

image_files = sorted(os.listdir(IMAGES_DIR))
mask_files = sorted(os.listdir(MASKS_DIR))

print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"CHECKPOINT_OUT = {CHECKPOINT_OUT}")
print(f"Found {len(image_files)} images")
print(f"Found {len(mask_files)} masks")
print(f"EVAL_ONLY      = {EVAL_ONLY}")


DATA_ROOT      = /content/drive/MyDrive/Kalana
CHECKPOINT_OUT = /content/drive/MyDrive/segformer_b0_scratch.pt
Found 5108 images
Found 5108 masks
EVAL_ONLY      = True


In [4]:
def mask_name_to_image_name(mask_filename):
    """Converts '855_mask_01.jpg' -> '855_sat_01.jpg'."""
    return mask_filename.replace("_mask", "_sat")

# Full-dataset pairing check before we trust it on everything
missing = []
for m in mask_files:
    expected_image = mask_name_to_image_name(m)
    if expected_image not in image_files:
        missing.append((m, expected_image))

print(f"Total masks: {len(mask_files)}")
print(f"Missing matches: {len(missing)}")
if missing:
    print("First few missing pairs:", missing[:5])
assert len(missing) == 0, "Fix missing pairs before continuing."

Total masks: 5108
Missing matches: 0


## Step 2: Dataset class

Same logic verified locally — resize, normalize, binarize the mask.

In [5]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

class ForestSegDataset(Dataset):
    def __init__(self, mask_filenames, images_dir, masks_dir, img_size=256):
        self.mask_filenames = mask_filenames
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size

        # ImageNet normalization stats (what SegFormer's pretrained backbone expects)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_fname = self.mask_filenames[idx]
        image_fname = mask_name_to_image_name(mask_fname)

        image = Image.open(self.images_dir / image_fname).convert("RGB")
        image = image.resize((self.img_size, self.img_size))

        mask = Image.open(self.masks_dir / mask_fname).convert("L")
        mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        # White (>127) = forest = 1, black = non-forest = 0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()

        return image, mask

## Step 3: Train / val / test split

In [6]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_splits():
    all_files = sorted(mask_files)
    random.shuffle(all_files)

    n = len(all_files)
    n_val = int(n * VAL_SPLIT)
    n_test = int(n * TEST_SPLIT)

    val_files = all_files[:n_val]
    test_files = all_files[n_val:n_val + n_test]
    train_files = all_files[n_val + n_test:]
    return train_files, val_files, test_files

set_seed(SEED)
train_files, val_files, test_files = make_splits()
print(f"Train/Val/Test sizes: {len(train_files)}/{len(val_files)}/{len(test_files)}")

Train/Val/Test sizes: 3576/766/766


## Step 4: Model + Table 2 metrics helpers

SegFormer-B0 architecture from scratch. Metrics for the forest (positive) class:
Dice, IoU, Precision, Recall, F1. Efficiency: Params + GFLOPs.


In [7]:
from transformers import SegformerConfig, SegformerForSemanticSegmentation
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

MIT_B0_CONFIG = dict(
    num_channels=3,
    num_encoder_blocks=4,
    depths=[2, 2, 2, 2],
    sr_ratios=[8, 4, 2, 1],
    hidden_sizes=[32, 64, 160, 256],
    num_attention_heads=[1, 2, 5, 8],
    patch_sizes=[7, 3, 3, 3],
    strides=[4, 2, 2, 2],
    mlp_ratios=[4, 4, 4, 4],
    decoder_hidden_size=256,
)

def build_model():
    """SegFormer-B0 with randomly initialized encoder + decode head."""
    cfg = SegformerConfig(
        num_labels=2,
        id2label={0: "non_forest", 1: "forest"},
        label2id={"non_forest": 0, "forest": 1},
        **MIT_B0_CONFIG,
    )
    return SegformerForSemanticSegmentation(cfg).to(device)

def confusion_counts(pred_mask, true_mask):
    """pred/true: bool tensors. Returns TP, FP, FN, TN as Python ints."""
    tp = (pred_mask & true_mask).sum().item()
    fp = (pred_mask & ~true_mask).sum().item()
    fn = (~pred_mask & true_mask).sum().item()
    tn = (~pred_mask & ~true_mask).sum().item()
    return tp, fp, fn, tn

def scores_from_counts(tp, fp, fn, eps=1e-6):
    """Micro-averaged forest-class scores from global counts."""
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    f1 = precision * recall * 2.0 / (precision + recall + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))

def estimate_gflops(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE)):
    """GFLOPs via thop; returns None if unavailable."""
    try:
        from thop import profile
        m = model
        was_training = m.training
        m.eval()

        class _Wrap(torch.nn.Module):
            def __init__(self, inner):
                super().__init__()
                self.inner = inner
            def forward(self, x):
                return self.inner(pixel_values=x).logits

        wrap = _Wrap(m).to(device)
        dummy = torch.randn(*input_size, device=device)
        flops, _ = profile(wrap, inputs=(dummy,), verbose=False)
        if was_training:
            m.train()
        return float(flops) / 1e9
    except Exception as e:
        print(f"GFLOPs estimate failed ({e}); will leave as n/a")
        return None

_m = build_model()
PARAMS = count_params(_m)
GFLOPS = estimate_gflops(_m)
print("From-scratch SegFormer-B0 ready")
print(f"  params = {PARAMS:,}  ({PARAMS/1e6:.2f}M)")
print(f"  GFLOPs = {GFLOPS if GFLOPS is not None else 'n/a'}  @ 1x3x{IMG_SIZE}x{IMG_SIZE}")
del _m
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Running on: cuda
From-scratch SegFormer-B0 ready
  params = 3,714,658  (3.71M)
  GFLOPs = 1.69213952  @ 1x3x256x256


## Step 5: DataLoaders + training loop

Training uses cross-entropy (same as the pretrained baseline). Validation tracks
Dice for checkpointing. Final Table 2 numbers come from **micro-averaged**
test TP/FP/FN in Step 6.


In [8]:
from torch.utils.data import DataLoader
from tqdm import tqdm

train_ds = ForestSegDataset(train_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
val_ds = ForestSegDataset(val_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
test_ds = ForestSegDataset(test_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)

nw = 0 if not IN_COLAB else 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=nw)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=nw)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=nw)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(f"train/val/test batches: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")


train/val/test batches: 447/96/96


In [9]:
def run_epoch(model, loader, optimizer=None):
    """Train/val loop. Returns loss, batch-mean dice, batch-mean iou, micro dict."""
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, total_dice, total_iou, n_batches = 0.0, 0.0, 0.0, 0
    tp = fp = fn = 0

    with torch.set_grad_enabled(is_train):
        for images, masks in tqdm(loader, leave=False):
            images, masks = images.to(device), masks.to(device)

            outputs = model(pixel_values=images)
            logits = F.interpolate(
                outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )
            loss = F.cross_entropy(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1).bool()
            true = masks.bool()
            tpc, fpc, fnc, _ = confusion_counts(preds, true)
            tp += tpc; fp += fpc; fn += fnc
            s = scores_from_counts(tpc, fpc, fnc)

            total_loss += loss.item()
            total_dice += s["dice"]
            total_iou += s["iou"]
            n_batches += 1

    micro = scores_from_counts(tp, fp, fn)
    return (
        total_loss / max(n_batches, 1),
        total_dice / max(n_batches, 1),
        total_iou / max(n_batches, 1),
        micro,
    )


@torch.no_grad()
def evaluate_table2(model, loader):
    """Full test pass -> micro-averaged Dice/IoU/F1/Precision (+ loss)."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    tp = fp = fn = 0
    for images, masks in tqdm(loader, leave=False, desc="test"):
        images, masks = images.to(device), masks.to(device)
        outputs = model(pixel_values=images)
        logits = F.interpolate(
            outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        total_loss += F.cross_entropy(logits, masks).item()
        n_batches += 1
        preds = logits.argmax(dim=1).bool()
        tpc, fpc, fnc, _ = confusion_counts(preds, masks.bool())
        tp += tpc; fp += fpc; fn += fnc
    metrics = scores_from_counts(tp, fp, fn)
    metrics["loss"] = total_loss / max(n_batches, 1)
    metrics["tp"], metrics["fp"], metrics["fn"] = int(tp), int(fp), int(fn)
    return metrics


In [10]:
best_val_dice = 0.0

if EVAL_ONLY:
    print(f"EVAL_ONLY: loading existing checkpoint {CHECKPOINT_OUT}")
    state = torch.load(CHECKPOINT_OUT, map_location=device)
    if isinstance(state, dict) and "model_state" in state:
        model.load_state_dict(state["model_state"])
    else:
        model.load_state_dict(state)
else:
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_dice, train_iou, _ = run_epoch(model, train_loader, optimizer)
        val_loss, val_dice, val_iou, val_micro = run_epoch(model, val_loader)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} dice={train_dice:.4f} iou={train_iou:.4f} | "
            f"val_loss={val_loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f} "
            f"(micro_f1={val_micro['f1']:.4f})"
        )

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), CHECKPOINT_OUT)
            print(f"  -> saved new best checkpoint ({CHECKPOINT_OUT})")


EVAL_ONLY: loading existing checkpoint /content/drive/MyDrive/segformer_b0_scratch.pt


## Step 6: Final test-set evaluation (Table 2 row)

Loads the best checkpoint and reports micro-averaged **Dice / IoU / F1 / Precision**,
plus **Params / GFLOPs** for the scratch SegFormer-B0 baseline.


In [11]:
import json

state = torch.load(CHECKPOINT_OUT, map_location=device)
if isinstance(state, dict) and "model_state" in state:
    model.load_state_dict(state["model_state"])
else:
    model.load_state_dict(state)

test_m = evaluate_table2(model, test_loader)
params = count_params(model)
gflops = estimate_gflops(model)

row = {
    "model": "SegFormer-B0 (from scratch)",
    "dice": round(test_m["dice"], 4),
    "iou": round(test_m["iou"], 4),
    "f1": round(test_m["f1"], 4),
    "precision": round(test_m["precision"], 4),
    "recall": round(test_m["recall"], 4),
    "params": params,
    "params_M": round(params / 1e6, 2),
    "gflops": None if gflops is None else round(gflops, 3),
    "checkpoint": str(CHECKPOINT_OUT),
    "seed": SEED,
    "epochs": EPOCHS,
    "img_size": IMG_SIZE,
}

print("\n========== TABLE 2 — SegFormer-B0 (scratch) ==========")
print(f"Dice       : {row['dice']:.4f}")
print(f"IoU        : {row['iou']:.4f}")
print(f"F1         : {row['f1']:.4f}")
print(f"Precision  : {row['precision']:.4f}")
print(f"Params     : {row['params_M']:.2f}M  ({row['params']:,})")
print(f"GFLOPs     : {row['gflops'] if row['gflops'] is not None else 'n/a'}")
print("====================================================")
gflops_str = "n/a" if row["gflops"] is None else f"{row['gflops']}"
print(
    f"| SegFormer-B0 (scratch) | {row['dice']:.4f} | {row['iou']:.4f} | "
    f"{row['f1']:.4f} | {row['precision']:.4f} | {row['params_M']:.1f}M | {gflops_str} |"
)

out_json = RESULTS_DIR / "table2_segformer_scratch.json"
out_txt = RESULTS_DIR / "table2_segformer_scratch.txt"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, indent=2)
with open(out_txt, "w", encoding="utf-8") as f:
    f.write(
        f"Model: {row['model']}\n"
        f"Dice: {row['dice']:.4f}\n"
        f"IoU: {row['iou']:.4f}\n"
        f"F1: {row['f1']:.4f}\n"
        f"Precision: {row['precision']:.4f}\n"
        f"Params: {row['params_M']:.2f}M\n"
        f"GFLOPs: {row['gflops']}\n"
    )
print(f"\nWrote {out_json}")
print(f"Wrote {out_txt}")



========== TABLE 2 — SegFormer-B0 (scratch) ==========
Dice       : 0.8511
IoU        : 0.7409
F1         : 0.8511
Precision  : 0.8066
Params     : 3.71M  (3,714,658)
GFLOPs     : 1.692
| SegFormer-B0 (scratch) | 0.8511 | 0.7409 | 0.8511 | 0.8066 | 3.7M | 1.692 |

Wrote /content/drive/MyDrive/segformer_scratch_results/table2_segformer_scratch.json
Wrote /content/drive/MyDrive/segformer_scratch_results/table2_segformer_scratch.txt


In [12]:
# Done. On Colab GPU, Run All to fill Table 2.
# If CHECKPOINT_OUT already exists, training is skipped (EVAL_ONLY=True).
